# 311 synthetic data and evaluation design

This notebook creates fictional resident reports, 311 question-answering examples, preference pairs, and attack probes. By default, 80% of reports contain fake personal details; the other 20% contain none. No real resident data is used.

The 1,000/200/200 train/validation/test sizes and attack-query counts follow the reference project's documented settings. The attack probes target private values in the training split, using validation wording for development and separate wording for the final test.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from generate_data import build_dataset, write_dataset

SEED = 42
dataset = build_dataset(seed=SEED)
output_dir = PROJECT_ROOT / "data" / "synthetic"
write_dataset(output_dir, dataset)
for name, rows in dataset.items():
    print(f"{name}: {len(rows)} rows")

## Compare the training conditions

- **Baseline:** all 1,000 report conversations, including those with PII, plus general 311 Q&A.
- **SFT filtering:** only reports without PII, plus the same general 311 Q&A. PII-containing report examples are omitted, not replaced with redaction markers.

Both models use resident messages as context but the training loss is computed only on assistant replies.

In [ ]:
from pprint import pprint

for split_name in ("training_records", "validation_records", "test_records"):
    records = dataset[split_name]
    pii_count = sum(record["contains_pii"] for record in records)
    print(f"{split_name}: {pii_count}/{len(records)} contain PII")

pprint(dataset["training_conversations"][0])
pprint(dataset["sft_filtered_training_conversations"][0])
pprint(dataset["dpo_training_pairs"][0])

## Evaluation set sizes

The test has 350 privacy probes: 100 each direct, indirect, and contextual, plus 50 jailbreak prompts. They target training records so the model has had a chance to learn the details. Validation probes use separate phrasings. Utility questions are separated into train, validation, and test versions.

In [ ]:
from collections import Counter

print("Validation attacks:", Counter(p["attack_type"] for p in dataset["validation_privacy_probes"]))
print("Test attacks:", Counter(p["attack_type"] for p in dataset["test_privacy_probes"]))
print("Training utility questions:", len(dataset["utility_train_qa"]))
print("Validation utility questions:", len(dataset["utility_validation_qa"]))
print("Test utility questions:", len(dataset["utility_test_qa"]))
pprint(dataset["test_privacy_probes"][0])
pprint(dataset["utility_test_qa"][0])

## Reproducibility checks

In [ ]:
assert dataset == build_dataset(seed=SEED)
train_ids = {row["record_id"] for row in dataset["training_records"]}
validation_ids = {row["record_id"] for row in dataset["validation_records"]}
test_ids = {row["record_id"] for row in dataset["test_records"]}
assert train_ids.isdisjoint(validation_ids | test_ids)
assert validation_ids.isdisjoint(test_ids)
assert sum(row["contains_pii"] for row in dataset["training_records"]) == 800
assert len(dataset["sft_filtered_training_conversations"]) == 200 + len(dataset["utility_train_qa"])
assert all(not row["contains_pii"] for row in dataset["sft_filtered_training_conversations"])
assert len(dataset["test_privacy_probes"]) == 350
assert all(probe["target_record_id"] in train_ids for probe in dataset["test_privacy_probes"])
print("All data-generation checks passed.")